**Studi Awal Smart Farming pada Komoditas Maize**

Tujuan utama notebook ini bukan untuk langsung menentukan judul skripsi atau memilih metode penyelesaian masalah.

Notebook ini bertujuan untuk:
1.   Merangkum temuan penting dari hasil analisis data Maize.
2.   Mengubah temuan data menjadi evidence atau bukti awal.
3.   Mengidentifikasi beberapa kandidat masalah penelitian.
4.   Menghubungkan kandidat masalah dengan kebutuhan informasi dalam konteks Sistem Informasi Bisnis.
5.   Mengidentifikasi informasi yang masih belum tersedia dari dataset.
6.   Menentukan hal-hal yang perlu divalidasi melalui mitra atau kondisi lapangan.
7.   Menentukan kandidat masalah yang layak dibawa ke tahap Literature Review.

Tahap ini penting karena data yang menunjukkan suatu pola belum otomatis berarti terdapat masalah bisnis.

Sebagai contoh, apabila terdapat perbedaan rata-rata hasil panen berdasarkan jenis irigasi, hal tersebut merupakan temuan data. Namun, belum dapat langsung disimpulkan bahwa salah satu metode irigasi lebih baik atau bahwa petani memiliki masalah dalam memilih metode irigasi.

Oleh karena itu, hasil EDA pada notebook ini diperlakukan sebagai bukti awal untuk membangun hipotesis masalah, bukan sebagai bukti final mengenai kondisi mitra.

> Pendekatan ini mengikuti prinsip bahwa penelitian Sistem Informasi Bisnis perlu dimulai dari masalah atau kebutuhan bisnis, kemudian dihubungkan dengan data, research gap, research question, solusi, dan evaluasi.

Untuk menjawab pertanyaan tersebut, analisis dilakukan dengan membedakan empat hal:

1. Temuan data = Apa yang benar-benar terlihat dari dataset?

2. Interpretasi = Apa arti temuan tersebut secara analitis?

3. Kandidat masalah = Apakah temuan tersebut dapat mengarah pada persoalan yang perlu diteliti?

4. Validasi = Apakah persoalan tersebut benar-benar terjadi pada mitra atau konteks penelitian?

# **Persiapan Data**

In [ ]:
# Import library yang diperlukan

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("Library berhasil dimuat.")

Library berhasil dimuat.


In [ ]:
# Menghubungkan Google Drive

from google.colab import drive

drive.mount('/content/drive')

print("Google Drive berhasil terhubung.")

Mounted at /content/drive
Google Drive berhasil terhubung.


In [ ]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

file_name = next(iter(uploaded))
maize = pd.read_csv(file_name)

print("Dataset berhasil dibaca.")
print("Jumlah baris :", maize.shape[0])
print("Jumlah kolom :", maize.shape[1])

Saving maize_cleaned_from_data_understanding.csv to maize_cleaned_from_data_understanding.csv
Dataset berhasil dibaca.
Jumlah baris : 500
Jumlah kolom : 22


In [ ]:
# Verifikasi dasar dataset

print("Jumlah observasi:", len(maize))
print("\nCrop type:")
print(maize['crop_type'].value_counts(dropna=False))

print("\nNama kolom:")
print(maize.columns.tolist())

Jumlah observasi: 500

Crop type:
crop_type
Maize      111
Soybean    108
Cotton     107
Wheat       92
Rice        82
Name: count, dtype: int64

Nama kolom:
['farm_id', 'region', 'crop_type', 'soil_moisture_%', 'soil_pH', 'temperature_C', 'rainfall_mm', 'humidity_%', 'sunlight_hours', 'irrigation_type', 'fertilizer_type', 'pesticide_usage_ml', 'sowing_date', 'harvest_date', 'total_days', 'yield_kg_per_hectare', 'sensor_id', 'timestamp', 'latitude', 'longitude', 'NDVI_index', 'crop_disease_status']


In [ ]:
# Memastikan dataset hanya berisi Maize

if maize['crop_type'].nunique() == 1 and maize['crop_type'].iloc[0] == 'Maize':
    print("Verifikasi berhasil: dataset berisi Maize.")
else:
    print("Perlu diperiksa kembali: terdapat crop type selain Maize.")

Perlu diperiksa kembali: terdapat crop type selain Maize.


Beberapa catatan kualitas data yang perlu dibawa ke tahap identifikasi masalah adalah:
*   Tidak ditemukan duplikasi data.
*   Variabel numerik utama tidak memiliki missing value.
*   irrigation_type memiliki data yang hilang.
*   crop_disease_status memiliki data yang hilang.
*   total_days konsisten dengan selisih antara harvest_date dan sowing_date.
*   Ukuran kelompok jenis irigasi tidak sama.
*   Dataset memiliki variabel aktual harvest_date, sehingga perlu berhati-hati ketika membicarakan "prediksi panen".




Kondisi tersebut penting karena keterbatasan data dapat memengaruhi jenis pertanyaan penelitian yang realistis untuk diajukan.

In [ ]:
# Ringkasan missing value

missing_summary = pd.DataFrame({
    'Missing_Count': maize.isna().sum(),
    'Missing_Percentage': (maize.isna().mean() * 100).round(2)
})

missing_summary = missing_summary[
    missing_summary['Missing_Count'] > 0
].sort_values('Missing_Count', ascending=False)

display(missing_summary)

,Missing_Count,Missing_Percentage
irrigation_type,150,30.0
crop_disease_status,130,26.0


In [ ]:
maize.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   farm_id               500 non-null    object 
 1   region                500 non-null    object 
 2   crop_type             500 non-null    object 
 3   soil_moisture_%       500 non-null    float64
 4   soil_pH               500 non-null    float64
 5   temperature_C         500 non-null    float64
 6   rainfall_mm           500 non-null    float64
 7   humidity_%            500 non-null    float64
 8   sunlight_hours        500 non-null    float64
 9   irrigation_type       350 non-null    object 
 10  fertilizer_type       500 non-null    object 
 11  pesticide_usage_ml    500 non-null    float64
 12  sowing_date           500 non-null    object 
 13  harvest_date          500 non-null    object 
 14  total_days            500 non-null    int64  
 15  yield_kg_per_hectare  5

In [ ]:
numeric_columns = maize.select_dtypes(include=np.number).columns

display(
    maize[numeric_columns].describe().round(2)
)

,soil_moisture_%,soil_pH,temperature_C,rainfall_mm,humidity_%,sunlight_hours,pesticide_usage_ml,total_days,yield_kg_per_hectare,latitude,longitude,NDVI_index
count,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00,500.00
mean,26.75,6.52,24.68,181.69,65.19,7.03,26.59,119.50,4032.93,22.44,80.39,0.60
std,10.15,0.59,5.35,72.29,14.64,1.69,13.20,16.80,1174.43,7.28,5.91,0.18
min,10.16,5.51,15.00,50.17,40.23,4.01,5.05,90.00,2023.56,10.00,70.02,0.30
25%,17.89,6.03,20.30,119.22,51.86,5.67,14.94,105.75,2994.82,16.26,75.37,0.45
50%,25.86,6.53,24.66,191.54,65.68,7.00,25.98,119.00,4071.69,21.98,80.65,0.61
75%,36.02,7.04,29.09,239.04,77.99,8.47,38.01,134.00,5062.11,28.53,85.65,0.75
max,44.98,7.50,34.84,298.96,90.00,10.00,49.94,150.00,5998.29,34.98,89.99,0.90


# **Kandidat Masalah 1 — Informasi Siklus Tanam dan Panen**

### Temuan awal

Dataset menyediakan:

* `sowing_date`
* `harvest_date`
* `total_days`
* `yield_kg_per_hectare`

Variabel tersebut memungkinkan kita melihat hubungan deskriptif antara lama siklus tanam dan hasil panen.

### Hasil EDA

Pada Notebook 2 diperoleh korelasi antara `total_days` dan `yield_kg_per_hectare` sebesar:

**-0.021301**

Nilai tersebut menunjukkan hubungan linear yang sangat lemah pada dataset ini.

### Interpretasi

Hasil tersebut menunjukkan bahwa pada dataset ini belum terdapat bukti kuat bahwa lama siklus tanam secara linear berhubungan dengan yield.

Namun, hal tersebut tidak berarti informasi siklus tanam dan panen tidak penting.

Masalah yang masih perlu ditanyakan adalah:

> Bagaimana informasi kondisi tanam dan umur tanaman digunakan untuk membantu menentukan atau memantau waktu panen?

Pertanyaan tersebut belum dapat dijawab hanya dari dataset.

Karena itu, kandidat masalah yang lebih aman pada tahap ini adalah:

> **Kebutuhan informasi untuk pemantauan siklus tanam dan panen.**

Bukan:

> **Sistem prediksi waktu panen.**

Alasannya adalah dataset sudah memiliki `harvest_date` aktual dan hasil EDA belum menunjukkan hubungan sederhana yang kuat antara `total_days` dan yield.

Kandidat ini masih perlu divalidasi dengan mitra.

In [ ]:
# Statistik siklus tanam

cycle_summary = maize['total_days'].describe()

display(cycle_summary.to_frame(name='Total_Days'))

,Total_Days
count,500.000000
mean,119.496000
std,16.798046
min,90.000000
25%,105.750000
50%,119.000000
75%,134.000000
max,150.000000


In [ ]:
# Korelasi total_days dengan yield

cycle_yield_corr = maize[
    ['total_days', 'yield_kg_per_hectare']
].corr().iloc[0, 1]

print(
    f"Korelasi Total Days dengan Yield: {cycle_yield_corr:.6f}"
)

Korelasi Total Days dengan Yield: -0.007570


# **Kandidat Masalah 2 — Informasi dan Evaluasi Irigasi**

### Temuan awal

Dataset menyediakan informasi:

* `irrigation_type`
* `soil_moisture_%`
* `rainfall_mm`
* `temperature_C`
* `humidity_%`
* `yield_kg_per_hectare`
* `total_days`

Dengan variabel tersebut kita dapat melakukan eksplorasi hubungan antara kondisi lingkungan, jenis irigasi, dan hasil pertanian.

### Hasil EDA

Jumlah observasi berdasarkan jenis irigasi pada data yang tersedia:

* Drip = 21
* Manual = 27
* Sprinkler = 32
* Missing = 31

Rata-rata yield:

* Drip = sekitar 4026.87 kg/ha
* Manual = sekitar 4209.96 kg/ha
* Sprinkler = sekitar 4111.59 kg/ha

### Interpretasi

Terdapat perbedaan nilai rata-rata yield antar kelompok irigasi.

Namun, perbedaan rata-rata tersebut belum cukup untuk menyatakan bahwa suatu jenis irigasi lebih efektif.

Beberapa alasan:

1. Ukuran kelompok berbeda.
2. Terdapat missing value pada jenis irigasi.
3. Dataset merupakan data awal untuk eksplorasi.
4. Perbedaan yield dapat dipengaruhi banyak variabel lain.
5. Analisis deskriptif tidak membuktikan hubungan sebab-akibat.

Karena itu, kandidat masalah yang lebih tepat adalah:

> **Kebutuhan informasi untuk pemantauan dan evaluasi kondisi irigasi dalam mendukung pengelolaan budidaya.**

Pertanyaan yang perlu divalidasi kepada mitra adalah apakah petani atau pengelola memang mengalami kesulitan dalam memperoleh informasi yang dibutuhkan untuk mengevaluasi irigasi.

Jika masalah tersebut benar-benar terjadi di lapangan, barulah dapat diteliti lebih lanjut mengenai bentuk informasi atau dukungan keputusan yang diperlukan.


In [ ]:
# Ringkasan jenis irigasi

irrigation_count = maize['irrigation_type'].value_counts(dropna=False)

display(irrigation_count)

,count
irrigation_type,
NaN,150
Sprinkler,121
Manual,118
Drip,111


In [ ]:
# Ringkasan yield berdasarkan jenis irigasi

irrigation_yield = (
    maize.groupby('irrigation_type', dropna=True)['yield_kg_per_hectare']
    .agg(['count', 'mean', 'median', 'std'])
    .round(2)
)

display(irrigation_yield)

,count,mean,median,std
irrigation_type,,,,
Drip,111,4019.45,4127.73,1240.98
Manual,118,4070.09,4117.35,1159.11
Sprinkler,121,4084.41,3997.67,1097.29


In [ ]:
# Ringkasan total_days berdasarkan jenis irigasi

irrigation_cycle = (
    maize.groupby('irrigation_type', dropna=True)['total_days']
    .agg(['count', 'mean', 'median', 'std'])
    .round(2)
)

display(irrigation_cycle)

,count,mean,median,std
irrigation_type,,,,
Drip,111,118.52,116.0,17.15
Manual,118,122.08,123.5,17.65
Sprinkler,121,118.32,117.0,16.68


# Kandidat Masalah 3 — Informasi Kondisi Tanaman

### Temuan awal

Dataset menyediakan:

* `NDVI_index`
* `crop_disease_status`
* `soil_moisture_%`
* `temperature_C`
* `humidity_%`
* `yield_kg_per_hectare`

Variabel tersebut dapat digunakan untuk mengeksplorasi kondisi tanaman.

### Hasil EDA

Korelasi antara NDVI dan yield pada dataset adalah sekitar:

**0.047351**

Hubungan tersebut sangat lemah.

Sementara itu, terdapat perbedaan rata-rata yield berdasarkan status penyakit:

* Mild ≈ 4210.48 kg/ha
* Moderate ≈ 3829.29 kg/ha
* Severe ≈ 3699.35 kg/ha

### Interpretasi

Data menunjukkan adanya variasi yield berdasarkan status penyakit.

Namun, hasil ini belum dapat digunakan untuk menyatakan bahwa penyakit tertentu menyebabkan penurunan yield karena terdapat kemungkinan pengaruh variabel lain.

Selain itu, `crop_disease_status` memiliki missing value.

Oleh karena itu, kandidat masalah yang lebih aman adalah:

> **Kebutuhan informasi mengenai kondisi tanaman sebagai bagian dari pemantauan budidaya.**

Kandidat ini dapat menjadi masalah pendukung, tetapi belum tentu menjadi fokus utama penelitian.

Kepentingannya harus ditentukan berdasarkan kebutuhan nyata mitra.

In [ ]:
# Distribusi status penyakit

disease_count = maize['crop_disease_status'].value_counts(dropna=False)

display(disease_count)

,count
crop_disease_status,
Severe,133
NaN,130
Mild,125
Moderate,112


In [ ]:
# Yield berdasarkan status penyakit

disease_yield = (
    maize.groupby('crop_disease_status', dropna=True)['yield_kg_per_hectare']
    .agg(['count', 'mean', 'median', 'std'])
    .round(2)
)

display(disease_yield)

,count,mean,median,std
crop_disease_status,,,,
Mild,125,4089.87,4127.73,1216.38
Moderate,112,3915.92,3912.78,1140.95
Severe,133,4029.34,4131.77,1210.50


In [ ]:
# Korelasi NDVI dengan yield

ndvi_yield_corr = maize[
    ['NDVI_index', 'yield_kg_per_hectare']
].corr().iloc[0, 1]

print(
    f"Korelasi NDVI dengan Yield: {ndvi_yield_corr:.6f}"
)

Korelasi NDVI dengan Yield: 0.038099


# Matriks Kandidat Masalah

Setelah temuan data dirangkum, langkah berikutnya adalah membuat matriks kandidat masalah.

Matriks ini digunakan agar proses pemilihan masalah tidak dilakukan hanya berdasarkan intuisi.

Setiap kandidat dilihat dari beberapa aspek:

* evidence dari data,
* keterbatasan evidence,
* kemungkinan kebutuhan Sistem Informasi Bisnis,
* ketersediaan data,
* dan kebutuhan validasi mitra.

Perlu diperhatikan bahwa kolom **"potensi masalah SI Bisnis" bukan berarti masalah tersebut sudah terbukti terjadi**.

Kolom tersebut hanya menunjukkan kemungkinan arah penelitian yang masih harus divalidasi.

Dengan demikian, matriks ini berfungsi sebagai alat untuk menyusun **hipotesis masalah penelitian**, bukan sebagai keputusan final.

In [ ]:
# Matriks kandidat masalah penelitian

candidate_problems = pd.DataFrame({
    'Kandidat': [
        'Siklus tanam dan panen',
        'Irigasi',
        'Kondisi tanaman'
    ],

    'Evidence_Data': [
        'Dataset memiliki sowing_date, harvest_date, total_days, dan yield. Korelasi total_days-yield sangat lemah.',
        'Terdapat irrigation_type, soil_moisture, rainfall, dan yield. Terdapat variasi yield antarjenis irigasi.',
        'Terdapat NDVI dan crop_disease_status. Status penyakit menunjukkan variasi yield, sedangkan korelasi NDVI-yield sangat lemah.'
    ],

    'Keterbatasan_Evidence': [
        'Belum membuktikan adanya masalah penentuan waktu panen pada mitra.',
        '31 data irrigation_type missing dan ukuran kelompok berbeda.',
        '36 data crop_disease_status missing dan hubungan sebab-akibat belum terbukti.'
    ],

    'Potensi_Masalah_SI_Bisnis': [
        'Kebutuhan informasi untuk pemantauan siklus tanam dan panen.',
        'Kebutuhan informasi untuk pemantauan dan evaluasi kondisi irigasi.',
        'Kebutuhan informasi untuk pemantauan kondisi tanaman.'
    ],

    'Data_Tersedia': [
        'Ya',
        'Sebagian',
        'Sebagian'
    ],

    'Perlu_Validasi_Mitra': [
        'Ya',
        'Ya',
        'Ya'
    ]
})

display(candidate_problems)

,Kandidat,Evidence_Data,Keterbatasan_Evidence,Potensi_Masalah_SI_Bisnis,Data_Tersedia,Perlu_Validasi_Mitra
0,Siklus tanam dan panen,"Dataset memiliki sowing_date, harvest_date, to...",Belum membuktikan adanya masalah penentuan wak...,Kebutuhan informasi untuk pemantauan siklus ta...,Ya,Ya
1,Irigasi,"Terdapat irrigation_type, soil_moisture, rainf...",31 data irrigation_type missing dan ukuran kel...,Kebutuhan informasi untuk pemantauan dan evalu...,Sebagian,Ya
2,Kondisi tanaman,Terdapat NDVI dan crop_disease_status. Status ...,36 data crop_disease_status missing dan hubung...,Kebutuhan informasi untuk pemantauan kondisi t...,Sebagian,Ya


# **Interpretasi Matriks Kandidat**

Matriks di atas menunjukkan bahwa ketiga kandidat memiliki evidence awal dari dataset.

Namun, evidence tersebut memiliki tingkat kekuatan dan keterbatasan yang berbeda.

*   **Kandidat A — Siklus Tanam dan Panen = Data tersedia cukup lengkap.**

    > Namun, EDA belum menunjukkan hubungan kuat antara lama siklus tanam dan yield.

    > Karena itu, fokus yang lebih aman adalah **kebutuhan informasi dan pemantauan siklus**, bukan langsung membuat model prediksi panen.

*   **Kandidat B — Irigasi = Dataset punya beberapa variabel yang relevan dengan irigasi.**

    > Terdapat variasi nilai yield antarjenis irigasi.

    > Namun, terdapat missing value dan ukuran kelompok yang berbeda.

    > Karena itu, belum boleh menyimpulkan metode irigasi tertentu sebagai metode terbaik.

    > Kandidat ini menarik untuk diteliti apabila mitra memang membutuhkan dukungan informasi dalam pengelolaan atau evaluasi irigasi.

*   **Kandidat C — Kondisi Tanaman = Dataset ada variabel NDVI dan status penyakit.**

    > Namun, hubungan NDVI dengan yield sangat lemah dan terdapat missing value pada status penyakit.

    > Kandidat ini lebih tepat diposisikan sebagai informasi pendukung sampai ada bukti kebutuhan yang lebih kuat dari mitra.

*   **Kesimpulan tahap ini**

Ketiga kandidat **belum dapat dinyatakan sebagai masalah penelitian final**.

Ketiganya merupakan kandidat yang perlu dibandingkan dengan:

1. kebutuhan nyata mitra,
2. literatur,
3. penelitian terdahulu,
4. research gap,
5. ketersediaan data,
6. dan kemungkinan evaluasi solusi.

# Keterbatasan Dataset pada Tahap Identifikasi Masalah

Dataset yang digunakan pada tahap awal memiliki beberapa keterbatasan yang harus dicatat secara eksplisit.

### **1. Dataset bukan bukti langsung kondisi mitra**

Dataset digunakan untuk eksplorasi awal dan identifikasi kemungkinan masalah.

Oleh karena itu, hasil analisis dataset tidak boleh langsung digeneralisasikan sebagai kondisi petani atau mitra penelitian.

### **2. Missing value**

Terdapat missing value pada:

* `irrigation_type`
* `crop_disease_status`

Kondisi ini membatasi interpretasi beberapa analisis.

### **3. Ukuran kelompok irigasi tidak seimbang**

Jumlah data untuk setiap jenis irigasi berbeda.

Hal tersebut perlu diperhatikan apabila nantinya dilakukan analisis perbandingan.

### **4. Korelasi bukan sebab-akibat**

Nilai korelasi hanya menunjukkan hubungan statistik tertentu pada dataset.

Korelasi tidak membuktikan bahwa perubahan suatu variabel menyebabkan perubahan variabel lain.

### **5. Harvest date merupakan informasi aktual**

Dataset memiliki `harvest_date`.

Oleh karena itu, jika penelitian nantinya menggunakan konsep "prediksi waktu panen", harus diperhatikan dengan serius bagaimana target penelitian didefinisikan agar informasi masa depan tidak digunakan secara tidak tepat.

### **6. Masalah bisnis belum tersedia di dataset**

Dataset memberikan data pertanian.

Namun, informasi mengenai:

* siapa pengguna,
* bagaimana proses keputusan dilakukan,
* sistem yang sedang digunakan,
* kesulitan pengguna,
* waktu proses,
* dampak biaya,
* dan kebutuhan informasi

belum tersedia.

Karena itu, validasi mitra tetap diperlukan.

In [ ]:
# Tabel ringkasan kandidat masalah

final_matrix = pd.DataFrame({
    'Kandidat Masalah': [
        'Informasi siklus tanam dan panen',
        'Informasi dan evaluasi irigasi',
        'Informasi kondisi tanaman'
    ],

    'Temuan Awal': [
        'Dataset memiliki informasi sowing, harvest, dan total_days; korelasi total_days-yield sangat lemah.',
        'Terdapat informasi jenis irigasi dan kondisi lingkungan; terdapat variasi yield antarjenis irigasi.',
        'Terdapat NDVI dan status penyakit; terdapat variasi yield berdasarkan status penyakit.'
    ],

    'Keterbatasan': [
        'Belum ada bukti bahwa mitra mengalami masalah dalam menentukan waktu panen.',
        'Missing irrigation_type dan ukuran kelompok tidak sama.',
        'Missing disease status dan hubungan sebab-akibat belum terbukti.'
    ],

    'Kebutuhan Validasi': [
        'Proses penentuan dan pemantauan waktu panen.',
        'Proses pengambilan keputusan dan pemantauan irigasi.',
        'Proses pemantauan kondisi dan penyakit tanaman.'
    ]
})

display(final_matrix)

,Kandidat Masalah,Temuan Awal,Keterbatasan,Kebutuhan Validasi
0,Informasi siklus tanam dan panen,"Dataset memiliki informasi sowing, harvest, da...",Belum ada bukti bahwa mitra mengalami masalah ...,Proses penentuan dan pemantauan waktu panen.
1,Informasi dan evaluasi irigasi,Terdapat informasi jenis irigasi dan kondisi l...,Missing irrigation_type dan ukuran kelompok ti...,Proses pengambilan keputusan dan pemantauan ir...
2,Informasi kondisi tanaman,Terdapat NDVI dan status penyakit; terdapat va...,Missing disease status dan hubungan sebab-akib...,Proses pemantauan kondisi dan penyakit tanaman.


# Kesimpulan Sementara

Berdasarkan Data Understanding, Data Quality, dan EDA terhadap dataset Maize, terdapat tiga kelompok kandidat masalah yang dapat dibawa ke tahap penelitian lebih lanjut:

### Kandidat 1 — Siklus Tanam dan Panen

Dataset menyediakan informasi mengenai tanggal tanam, tanggal panen, lama siklus, dan hasil panen.

Namun, hubungan linear antara lama siklus dan yield sangat lemah.

Karena itu, fokus penelitian belum diarahkan langsung pada prediksi waktu panen.

Fokus yang masih terbuka adalah kebutuhan informasi untuk pemantauan siklus dan panen.

### Kandidat 2 — Irigasi

Dataset memiliki informasi mengenai jenis irigasi, kelembapan tanah, curah hujan, dan variabel lingkungan lainnya.

Terdapat variasi yield berdasarkan jenis irigasi.

Namun, terdapat missing value dan ukuran kelompok yang berbeda sehingga hasil tersebut belum dapat digunakan untuk menentukan metode irigasi terbaik.

Kandidat ini perlu divalidasi dengan mitra untuk mengetahui apakah terdapat kebutuhan nyata terkait pemantauan atau pengambilan keputusan irigasi.

### Kandidat 3 — Kondisi Tanaman

Dataset menyediakan NDVI dan status penyakit tanaman.

Terdapat variasi yield berdasarkan status penyakit, tetapi hubungan NDVI dan yield sangat lemah.

Kandidat ini dapat menjadi informasi pendukung untuk pemantauan kondisi tanaman, tetapi belum cukup kuat untuk dijadikan fokus utama tanpa validasi kebutuhan mitra.

### Kesimpulan utama

**Belum ada masalah penelitian final yang ditetapkan pada Notebook 3.**

Tahap ini menghasilkan **kandidat masalah dan evidence awal**.

Penetapan masalah penelitian akan dilakukan setelah:

1. validasi kebutuhan mitra,
2. pencarian literatur,
3. identifikasi penelitian terdahulu,
4. identifikasi keterbatasan penelitian terdahulu,
5. penyusunan research gap,
6. penyusunan research question,
7. dan penentuan indikator evaluasi.

Dengan demikian, penelitian tidak dimulai dari asumsi bahwa solusi tertentu sudah pasti diperlukan.

# Simpan

In [ ]:
# Export matriks kandidat masalah

output_candidate = '/content/kandidat_masalah_maize.csv'

final_matrix.to_csv(
    output_candidate,
    index=False
)

print("Matriks kandidat masalah berhasil disimpan.")

from google.colab import files
files.download(output_candidate)

Matriks kandidat masalah berhasil disimpan.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>